<a href="https://colab.research.google.com/github/sourabh90/mlops-starter/blob/main/1_1_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### ML Pipelines with ZenML

Install ZenML Server, sklearn integration

In [1]:
%pip install "zenml[server]==0.80"
!zenml integration install sklearn -y

import IPython

# auto restart kernel
IPython.Application.instance().kernel.do_shutdown(restart=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.7/593.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.8/209.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.9/423.9 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{'status': 'ok', 'restart': True}

#### Install Python NGROK library and authenticate
NGROK will enable visualization

In [1]:
NGROK_TOKEN = '34CncgUkecxJtj8QgSAJ55Wy2br_4Awb6U7zaQnzWEBGPqida'

In [2]:
from zenml.environment import Environment

if Environment.in_google_colab():
    !pip install pyngrok
    !ngrok authtoken {NGROK_TOKEN}

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


#### Setup ZenML

In [3]:
# Remove any previous ZenML setup or config
!rm -rf .zen
!zenml init

Initializing the ZenML global configuration version to 0.80.0
⠋ Initializing ZenML repository at /content.
⠙ Initializing ZenML repository at /content.
⠸ Initializing ZenML repository at /content.
⠼ Initializing ZenML repository at /content.
⠴ Initializing ZenML repository at /content.
⠦ Initializing ZenML repository at /content.
⠧ Initializing ZenML repository at /content.
⠇ Initializing ZenML repository at /content.
⠏ Initializing ZenML repository at /content.
⠙ Initializing ZenML repository at /content.
⠹ Initializing ZenML repository at /content.
⠸ Initializing ZenML repository at /content.
⠼ Initializing ZenML repository at /content.
⠙ Initializing ZenML repository at /content.
⠹ Initializing ZenML repository at /content.
⠼ Initializing ZenML repository at /content.
⠦ Initializing ZenML repository at /content.
⠧ Initializing ZenML repository at /content.
⠇ Initializing ZenML repository at /content.
⠏ Initializing ZenML repository at /content.
⠋ Initializing ZenML repository at /co

#### Example ML Experimentation Code

In [4]:
import numpy as np
from sklearn.base import ClassifierMixin
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def train_test():
  '''Train and test a Scikei-learn SVC classifier on digits'''
  digits = load_digits()
  data = digits.images.reshape((len(digits.images), -1))
  X_train, X_test, y_train, y_test = train_test_split(
      data, digits.target, test_size=0.2, shuffle=False
  )
  model = SVC(gamma=0.001)
  model.fit(X_train, y_train)
  test_acc = model.score(X_test, y_test)
  print(f'Test accuracy: {test_acc}')

train_test()

INFO:numexpr.utils:NumExpr defaulting to 2 threads.


NumExpr defaulting to 2 threads.
Test accuracy: 0.9583333333333334


#### Turning ML into pipelines with ZenML


    Zen ML Repository
    
    First Pipeline
    
    Importer --> SVC Trainer --> Evaluator


We can identify 3 distinct steps in out example: data loading, model training and model evaluation. Let's define each of these steps as a ZenML pipeline step simply by moving each step to its own function and decorating them with ZenML's python decorator @step.



In [5]:
from zenml import step
from typing_extensions import Annotated
import pandas as pd
from typing import Tuple

@step
def importer() -> Tuple[
  Annotated[np.ndarray, 'X_train'],
  Annotated[np.ndarray, 'X_test'],
  Annotated[np.ndarray, 'y_train'],
  Annotated[np.ndarray, 'y_test'],
]:
  '''Load the digits datasets as numpy array'''
  digits = load_digits()
  data = digits.images.reshape((len(digits.images), -1))
  X_train, X_test, y_train, y_test = train_test_split(
      data, digits.target, test_size=0.2, shuffle=False
  )
  return X_train, X_test, y_train, y_test


@step
def svc_trainer(
  X_train: np.ndarray,
  y_train: np.ndarray,
) -> ClassifierMixin:
  '''Train an sklearn SVC classifier'''
  model = SVC(gamma=0.001)
  model.fit(X_train, y_train)
  return model


@step
def evaluator(
  X_test: np.ndarray,
  y_test: np.ndarray,
  model: ClassifierMixin
) -> float:
  '''Calculate the test set accuracy of an sklearn model'''
  test_acc = model.score(X_test, y_test)
  print(f'Test accuracy: {test_acc}')
  return test_acc



Similarly we can use ZenML's @pipeline decorator to connect all of our steps into an MLOps pipeline.

In [6]:
from zenml import pipeline

@pipeline
def digits_pipeline():
  '''Links all the steps together in a MLOps pipeline'''
  X_train, X_test, y_train, y_test = importer()
  model = svc_trainer(X_train, y_train)
  evaluator(X_test, y_test, model)


#### Running ZenML Pipelines


In [7]:
digits_svc_pipeline = digits_pipeline()

Initiating a new run for the pipeline: digits_pipeline.
Registered new pipeline: digits_pipeline.
Using user: default
Using stack: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step importer has started.
No materializer is registered for type <class 'numpy.ndarray'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'numpy.ndarray'> according to the instructions at https://docs.zenml.io/how-to/handle-data-artifacts/handle-custom-data-types
No materializer is registered for type <class 'numpy.ndarray'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loa

After running the pipeline you can visualize the dashboard.

In [ ]:
from zenml.environment import Environment

def start_zenml_dashboard(port=8237):
  if Environment.in_google_colab():
    from pyngrok import ngrok

    public_url = ngrok.connect(port)
    print(f'\x1b[31mIn Colab, use this URL instead: {public_url}!\x1b[0m')
    !zenml up --blocking --port {port}

  else:
    !zenml up --port {port}

start_zenml_dashboard()


INFO:pyngrok.ngrok:Opening tunnel named: http-8237-a46b73d0-9c56-4014-b6fc-ebf8d1031542


Opening tunnel named: http-8237-a46b73d0-9c56-4014-b6fc-ebf8d1031542


INFO:pyngrok.process.ngrok:t=2025-10-19T18:09:48+0000 lvl=info msg="no configuration paths supplied"


t=2025-10-19T18:09:48+0000 lvl=info msg="no configuration paths supplied"


INFO:pyngrok.process.ngrok:t=2025-10-19T18:09:48+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml


...

t=2025-10-19T18:17:53+0000 lvl=info msg="join connections" obj=join id=b375b130c8d4 l=127.0.0.1:8237 r=[2a01:4b00:8077:9900:7438:75d0:6b9b:7a68]:51691
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/components/eec213db-a634-4c67-a675-f2b44b6ba6a4 HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/artifact_versions/7badc10c-8c5b-45da-a694-3efcfc0a96f3 HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/steps/0c437c87-91ca-4afe-92db-17fdd601e67d HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/runs/363e33bc-8194-45ca-aaf3-a45f4d2a3116 HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/artifact_versions/7badc10c-8c5b-45da-a694-3efcfc0a96f3 HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/artifact_versions/7badc10c-8c5b-45da-a694-3efcfc0a96f3 HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a

INFO:pyngrok.process.ngrok:t=2025-10-19T18:17:55+0000 lvl=info msg="join connections" obj=join id=79cfe7358ded l=127.0.0.1:8237 r=[2a01:4b00:8077:9900:7438:75d0:6b9b:7a68]:51691


t=2025-10-19T18:17:55+0000 lvl=info msg="join connections" obj=join id=79cfe7358ded l=127.0.0.1:8237 r=[2a01:4b00:8077:9900:7438:75d0:6b9b:7a68]:51691


INFO:pyngrok.process.ngrok:t=2025-10-19T18:17:55+0000 lvl=info msg="join connections" obj=join id=ad64d9a0d341 l=127.0.0.1:8237 r=[2a01:4b00:8077:9900:7438:75d0:6b9b:7a68]:51691


INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/runs/363e33bc-8194-45ca-aaf3-a45f4d2a3116 HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/steps/0c437c87-91ca-4afe-92db-17fdd601e67d HTTP/1.1" 200 OK
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/components/eec213db-a634-4c67-a675-f2b44b6ba6a4 HTTP/1.1" 200 OK
t=2025-10-19T18:17:55+0000 lvl=info msg="join connections" obj=join id=ad64d9a0d341 l=127.0.0.1:8237 r=[2a01:4b00:8077:9900:7438:75d0:6b9b:7a68]:51691
INFO:     2a01:4b00:8077:9900:7438:75d0:6b9b:7a68:0 - "GET /api/v1/artifact_versions/7badc10c-8c5b-45da-a694-3efcfc0a96f3 HTTP/1.1" 200 OK
